# 1. Sentiment Identification

In [1]:
import pandas as pd
import re
import nltk

NLTK_AVAILABLE = {}
for resource, resource_path in {
    "punkt": "tokenizers/punkt",
    "averaged_perceptron_tagger": "taggers/averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng": "taggers/averaged_perceptron_tagger_eng",
    "wordnet": "corpora/wordnet",
    "stopwords": "corpora/stopwords",
}.items():
    try:
        nltk.data.find(resource_path)
        NLTK_AVAILABLE[resource] = True
    except LookupError:
        NLTK_AVAILABLE[resource] = False

print("NLTK resource status:", NLTK_AVAILABLE)

NLTK resource status: {'punkt': True, 'averaged_perceptron_tagger': True, 'averaged_perceptron_tagger_eng': False, 'wordnet': False, 'stopwords': True}


In [2]:
pain_points = '../output/challenges.csv'
expectations = '../output/expectations.csv'

df_pain_points = pd.read_csv(pain_points)
df_expectations = pd.read_csv(expectations)

In [3]:
df_pain_points.shape, df_expectations.shape

((424, 2), (433, 2))

In [4]:
# remove records with less than or equal to 4 words
df_pain_points = df_pain_points[df_pain_points['pain_points'].str.split().str.len() > 4]
df_expectations = df_expectations[df_expectations['expectations'].str.split().str.len() > 4]

In [5]:
df_pain_points.shape, df_expectations.shape

((324, 2), (309, 2))

In [6]:
df_pain_points.sample(5)

,focus_group,pain_points
324,DOCE Office Manager,Unexpected shutdowns can result in lost work.
167,DOCE Building Inspectors,Address Discrepancies: Property addresses are ...
126,DOCE Admin Aide,Case notes and attachments are not always orga...
143,DOCE Admin Aide,Multiple property research tools must be used ...
24,BAA BAA Supervisors,Unlinked Related Cases: Cases involving the sa...


In [7]:
from nltk import pos_tag, word_tokenize

def safe_word_tokenize(text):
    s = str(text)
    try:
        return word_tokenize(s)
    except LookupError:
        return re.findall(r"[A-Za-z]+", s)

def safe_pos_tag(tokens):
    try:
        return pos_tag(tokens)
    except LookupError:
        return [(tok, "NN") for tok in tokens]

def extract_keywords(text):
    tokens = safe_word_tokenize(text)
    tagged = safe_pos_tag(tokens)
    keywords = [
        word.lower()
        for word, tag in tagged
        if tag.startswith("NN") or tag.startswith("JJ")
    ]
    return keywords

In [8]:
from nltk import RegexpParser

grammar = r"""
NP:
    {<JJ>*<NN.*>+}
"""

chunker = RegexpParser(grammar)

df_pain_points["processed_text"] = (
    df_pain_points["pain_points"]
    .fillna("")
    .astype(str)
    .str.replace(r"[^A-Za-z\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

df_expectations["processed_text"] = (
    df_expectations["expectations"]
    .fillna("")
    .astype(str)
    .str.replace(r"[^A-Za-z\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

def extract_phrases(text):
    tokens = safe_word_tokenize(text)
    tagged = safe_pos_tag(tokens)
    tree = chunker.parse(tagged)

    phrases = []
    for subtree in tree.subtrees():
        if subtree.label() == "NP":
            phrase = " ".join(
                word.lower()
                for word, _ in subtree.leaves()
            )
            phrases.append(phrase)

    return phrases

df_pain_points.head(3)

,focus_group,pain_points,processed_text
1,Assessment Assessment Supervisors,Historical Record Management: Historical permi...,historical record management historical permit...
2,Assessment Assessment Supervisors,Incomplete Property History: IPS does not alwa...,incomplete property history ips does not alway...
3,Assessment Assessment Supervisors,Large Plan Viewing: Large plans and surveys ca...,large plan viewing large plans and surveys can...


In [9]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model = "cardiffnlp/twitter-roberta-base-sentiment-latest"
)

pain_point_preds = classifier(df_pain_points["processed_text"].tolist())
expectation_preds = classifier(df_expectations["processed_text"].tolist())

df_pain_points["sentiment"] = [pred["label"].lower() for pred in pain_point_preds]
df_expectations["sentiment"] = [pred["label"].lower() for pred in expectation_preds]

df_pain_points[["processed_text", "sentiment"]].head(3)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,processed_text,sentiment
1,historical record management historical permit...,neutral
2,incomplete property history ips does not alway...,neutral
3,large plan viewing large plans and surveys can...,neutral


In [10]:
df_pain_points[["processed_text", "sentiment"]].sample(3)

,processed_text,sentiment
333,no centralized source of property and ownershi...,neutral
79,permit to code disconnect code enforcement and...,neutral
20,ownership research ownership information requi...,neutral


In [11]:
# total pain points per focus group (show all groups)
pain_points_summary = df_pain_points.groupby("focus_group", dropna=False).agg(
    total_pain_points=pd.NamedAgg(column="pain_points", aggfunc="count"),
    negative_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "negative").sum()),
    positive_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "positive").sum()),
    neutral_pain_points=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "neutral").sum())
).reset_index()

# Validation: these two numbers should match
print("distinct focus groups in data:", df_pain_points["focus_group"].nunique(dropna=False))

pain_points_summary.sort_values(by=["total_pain_points", "focus_group"], ascending=[False, True])

distinct focus groups in data: 16


,focus_group,total_pain_points,negative_pain_points,positive_pain_points,neutral_pain_points
6,DOCE Admin Aide,34,14,0,20
8,DOCE CommercialPermitElectrical Inspectors,30,14,0,16
10,DOCE Housing Inspectors,30,11,0,19
7,DOCE Building Inspectors,29,18,0,11
11,DOCE Office Manager,28,9,0,19
12,DOCE Supervisors,26,10,0,16
9,DOCE Fire Prevention Bureau,25,12,0,13
4,CPO CPO Co-Ordinator,22,8,0,14
5,CPO Central Permit Office,21,7,0,14
15,NBD NBD Internal,14,4,0,10


In [12]:
# total expectations per focus group (show all groups)
expectations_summary = df_expectations.groupby("focus_group", dropna=False).agg(
    total_expectations=pd.NamedAgg(column="expectations", aggfunc="count"),
    negative_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "negative").sum()),
    positive_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "positive").sum()),
    neutral_expectations=pd.NamedAgg(column="sentiment", aggfunc=lambda x: (x == "neutral").sum())
).reset_index()

# Validation: these two numbers should match
print("distinct focus groups in data:", df_expectations["focus_group"].nunique(dropna=False))
expectations_summary.sort_values(by=["total_expectations", "focus_group"], ascending=[False, True])

distinct focus groups in data: 16


,focus_group,total_expectations,negative_expectations,positive_expectations,neutral_expectations
10,DOCE Housing Inspectors,31,2,2,27
12,DOCE Supervisors,30,1,5,24
7,DOCE Building Inspectors,27,0,3,24
5,CPO Central Permit Office,26,0,1,25
6,DOCE Admin Aide,26,2,1,23
8,DOCE CommercialPermitElectrical Inspectors,26,0,4,22
9,DOCE Fire Prevention Bureau,26,1,2,23
4,CPO CPO Co-Ordinator,25,0,6,19
11,DOCE Office Manager,21,1,2,18
3,CPC CPC,13,0,1,12


In [13]:
# display expectations that are negative
neg_expectations = df_expectations.loc[
    df_expectations["sentiment"] == "negative",
    ["focus_group", "expectations", "sentiment"]
]
neg_expectations.sample(n=min(10, len(neg_expectations)), random_state=42)

,focus_group,expectations,sentiment
133,DOCE Admin Aide,Reduce crashes and memory-related errors.,negative
152,DOCE Admin Aide,Resolve phone system issues causing dropped ca...,negative
324,DOCE Office Manager,Reduced crashes and memory issues.,negative
254,DOCE Fire Prevention Bureau,Modern User Interface: Replace outdated and no...,negative
277,DOCE Housing Inspectors,Simplified Dropdowns: Reduce unnecessary dropd...,negative
266,DOCE Housing Inspectors,Missing Inspection Tracking: Report on cases w...,negative
354,DOCE Supervisors,Missed Case Identification: Automatically iden...,negative


In [14]:
#convert negative sentiment to neutral for expectations
df_expectations.loc[df_expectations['sentiment'] == 'negative', 'sentiment'] = 'neutral'

In [15]:
#save as csv
df_pain_points.to_csv("../output/clean_challenges.csv", index=False)
df_expectations.to_csv("../output/clean_expectations.csv", index=False)

# 2. Categorize the records

### Extract Noun Phrases

In [16]:
grammar = r"""
CATEGORY:
    {<JJ><NN>}
    {<NN><NN><JJ>}
    # {<JJ><NN><NN>}
    # {<NN><IN><NN>}
    {<NN><NN><NN>}
"""

chunker = RegexpParser(grammar)
def extract_phrases(text):
    tokens = safe_word_tokenize(text)
    tagged = safe_pos_tag(tokens)
    tree = chunker.parse(tagged)

    phrases = []
    for subtree in tree.subtrees():
        if subtree.label() == "CATEGORY":
            phrase = " ".join(
                word.lower()
                for word, _ in subtree.leaves()
            )
            phrases.append(phrase)

    return phrases

### Count Phrase Frequency

In [17]:
from collections import Counter
phrase_counter = Counter()
for text in df_pain_points["processed_text"]:
    phrases = extract_phrases(text)
    phrase_counter.update(phrases)

raw_categories = (
    phrase_counter
    .most_common(10)
)

for cat in raw_categories:
    print(cat)

('duplicate data entry', 6)
('limited search functionality', 4)
('cross department dependencies', 4)
('information must be', 4)
('be difficult to', 4)
('difficult to locate', 4)
('it difficult to', 3)
('information is not', 3)
('ips camino as', 3)
('information is spread', 3)


In [18]:
from collections import Counter
from nltk.util import ngrams
from sklearn.feature_extraction.text import TfidfVectorizer

documents = df_pain_points["processed_text"].fillna("").astype(str).tolist()

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()
term_scores = tfidf_matrix.mean(axis=0).A1
word_weights = dict(zip(feature_names, term_scores))

def rank_weighted_ngrams(text_series, n, top_k=25):
    ngram_scores = Counter()
    ngram_counts = Counter()

    for text in text_series.fillna("").astype(str):
        tokens = [token for token in safe_word_tokenize(text) if token in word_weights]
        for gram in ngrams(tokens, n):
            score = sum(word_weights[token] for token in gram) / n
            ngram_scores[gram] += score
            ngram_counts[gram] += 1

    ranked = sorted(
        (
            (gram, round(ngram_scores[gram], 4), ngram_counts[gram])
            for gram in ngram_scores
        ),
        key=lambda item: (item[1], item[2]),
        reverse=True,
    )
    return ranked[:top_k]

top_bigrams = rank_weighted_ngrams(df_pain_points["processed_text"], 2)
top_trigrams = rank_weighted_ngrams(df_pain_points["processed_text"], 3)

print("Top weighted bigrams (ngram, score, count):")
for item in top_bigrams:
    print(item)

print("\nTop weighted trigrams (ngram, score, count):")
for item in top_trigrams:
    print(item)

Top weighted bigrams (ngram, score, count):
(('multiple', 'systems'), np.float64(0.4843), 15)
(('ownership', 'information'), np.float64(0.3483), 9)
(('information', 'difficult'), np.float64(0.3325), 8)
(('ips', 'camino'), np.float64(0.2485), 10)
(('property', 'information'), np.float64(0.2363), 6)
(('difficult', 'locate'), np.float64(0.2156), 10)
(('property', 'research'), np.float64(0.1885), 8)
(('data', 'entry'), np.float64(0.186), 12)
(('information', 'spread'), np.float64(0.1809), 6)
(('historical', 'records'), np.float64(0.1497), 7)
(('information', 'requires'), np.float64(0.134), 4)
(('information', 'easily'), np.float64(0.1282), 4)
(('information', 'entered'), np.float64(0.1212), 4)
(('information', 'exists'), np.float64(0.1209), 4)
(('historical', 'property'), np.float64(0.1185), 5)
(('multiple', 'locations'), np.float64(0.1175), 5)
(('historical', 'information'), np.float64(0.1096), 3)
(('manual', 'effort'), np.float64(0.1032), 6)
(('duplicate', 'data'), np.float64(0.1021), 6)

In [19]:
def select_weighted_keyword(text):
    tokens = [
        token
        for token in safe_word_tokenize(str(text).lower())
        if token in word_weights
    ]

    candidates = []

    for n in (3, 2):
        for gram in ngrams(tokens, n):
            score = sum(word_weights[token] for token in gram) / n
            candidates.append((" ".join(gram), score, n))

    for token in tokens:
        candidates.append((token, word_weights[token], 1))

    if not candidates:
        return ""

    best_keyword, _, _ = max(candidates, key=lambda item: (item[1], item[2]))
    return best_keyword

df_pain_points["keywords"] = df_pain_points["processed_text"].apply(select_weighted_keyword)

print(df_pain_points[["processed_text", "keywords"]].head(10))

                                       processed_text     keywords
1   historical record management historical permit...  information
2   incomplete property history ips does not alway...          ips
3   large plan viewing large plans and surveys can...          ips
4   limited historical visibility historical recor...     property
5   historical property research research often re...     property
6   resubdivision coordination parcel updates requ...      require
7   permit driven workload most assessment work be...       permit
9   seasonal review process permit monitoring and ...       permit
12  manual ticket review staff has to open individ...       manual
13  manual fee processing default fees must be app...       manual


In [21]:
print(df_pain_points[df_pain_points["keywords"].str.split().str.len() == 3].head(10))

         focus_group                                     pain_points  \
156  DOCE Admin Aide  Front-counter traffic can become overwhelming.   

                                    processed_text sentiment  \
156  front counter traffic can become overwhelming   neutral   

                         keywords  
156  counter traffic overwhelming  
